# 01b Prepare CNC Machining Dataset

This notebook downloads the Bosch Research CNC Machining repository if needed and runs the project preparation script to create windowed NPZ files and deterministic split manifests.

In [6]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

In [7]:
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

CNC_REPO_URL = "https://github.com/boschresearch/CNC_Machining.git"
CNC_SOURCE_ROOT = REPO_ROOT / "data" / "raw_cnc_machining"
OUTPUT_ROOT = REPO_ROOT / "data" / "01_windowed_labeled_cnc_machining"
MANIFEST_PATH = REPO_ROOT / "reports" / "manifests" / "cnc_machining_split_seed42.csv"
SUMMARY_PATH = REPO_ROOT / "reports" / "manifests" / "cnc_machining_split_summary.csv"

SEED = 42
LABEL_SCHEME = "anomaly"
OVERWRITE = True
NORMALIZE_CHANNELS = True

## Lazy Download

In [8]:
has_h5_files = CNC_SOURCE_ROOT.exists() and any(CNC_SOURCE_ROOT.rglob("*.h5"))

if has_h5_files:
    print("Using existing CNC source data:", CNC_SOURCE_ROOT)
elif CNC_SOURCE_ROOT.exists() and any(CNC_SOURCE_ROOT.iterdir()):
    raise FileExistsError(
        f"{CNC_SOURCE_ROOT} exists but contains no .h5 files. "
        "Move it aside or set CNC_SOURCE_ROOT to the CNC_Machining checkout."
    )
else:
    CNC_SOURCE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", CNC_REPO_URL, str(CNC_SOURCE_ROOT)], check=True)
    print("Cloned CNC source data to", CNC_SOURCE_ROOT)

Using existing CNC source data: /home/cgawron/git/spectrogram-anomaly-ae/data/raw_cnc_machining


## Prepare Windows

In [9]:
prepared_outputs_exist = MANIFEST_PATH.exists() and any(OUTPUT_ROOT.rglob("*.npz"))

if prepared_outputs_exist and not OVERWRITE:
    print("Using existing prepared CNC windows:", OUTPUT_ROOT)
    print("Manifest:", MANIFEST_PATH)
else:
    cmd = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "prepare_cnc_machining_dataset.py"),
        "--source-root", str(CNC_SOURCE_ROOT),
        "--output-root", str(OUTPUT_ROOT),
        "--manifest-path", str(MANIFEST_PATH),
        "--summary-path", str(SUMMARY_PATH),
        "--seed", str(SEED),
        "--label-scheme", LABEL_SCHEME,
    ]
    if NORMALIZE_CHANNELS:
        cmd.append("--normalize-channels")
    if OVERWRITE:
        cmd.append("--overwrite")

    subprocess.run(cmd, check=True, cwd=REPO_ROOT)


CNC channel normalization mean=[2.8912801715133516, 17.57863841567711, -1024.0667401446206] std=[398.41419177301503, 197.20792183703512, 196.3778215285657] samples=121214586
Wrote 1702 windows from 1702 CNC files
Manifest: /home/cgawron/git/spectrogram-anomaly-ae/reports/manifests/cnc_machining_split_seed42.csv
Summary: /home/cgawron/git/spectrogram-anomaly-ae/reports/manifests/cnc_machining_split_summary.csv


In [10]:
if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)
    display(manifest.groupby(["source_dataset", "split", "label"]).size().rename("n").reset_index())
else:
    raise FileNotFoundError(f"Manifest was not created: {MANIFEST_PATH}")

,source_dataset,split,label,n
0,cnc_machining,test,anomaly,35
1,cnc_machining,test,nominal,245
2,cnc_machining,train,nominal,1142
3,cnc_machining,validation,anomaly,35
4,cnc_machining,validation,nominal,245
